In [1]:
import numpy as np
import pandas as pd
import warnings
import ast
import re
import nltk
import os
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

In [2]:

BASE     = r"C:\Users\Rapid IT World\Desktop\movie_recommendations\data"
MOVIES   = os.path.join(BASE, "movies.csv")  
CREDITS  = os.path.join(BASE, "credits.csv")
KEYWORDS = os.path.join(BASE, "keywords.csv")
EMB_PATH = os.path.join(BASE, "embeddings.pkl")

In [3]:

# =========================================================
# LOAD
# =========================================================

print("Loading movies_metadata.csv ...")
df = pd.read_csv(MOVIES, low_memory=False)
print(f"  {len(df):,} rows")

Loading movies_metadata.csv ...
  45,466 rows


In [4]:
# Drop malformed id rows (the file has a few garbage rows)
df = df[pd.to_numeric(df["id"], errors="coerce").notna()].copy()
df["id"] = df["id"].astype(int)
df = df.drop_duplicates(subset="id").reset_index(drop=True)


In [5]:

KEEP = ["id", "title", "overview", "genres", "tagline",
        "vote_average", "popularity", "release_date", "poster_path"]
df = df[[c for c in KEEP if c in df.columns]]
df = df.dropna(subset=["title"])

for col, val in [("overview",""), ("tagline",""), ("genres","[]"),
                  ("release_date",""), ("poster_path","")]:
    df[col] = df[col].fillna(val)

df["vote_average"] = pd.to_numeric(df["vote_average"], errors="coerce").fillna(0)
df["popularity"]   = pd.to_numeric(df["popularity"],   errors="coerce").fillna(0)

In [6]:
# =========================================================
# PARSERS
# =========================================================

def parse_names(text, limit=None):
    """
    Parse a JSON-like list of dicts and return names as single tokens.
    'Tom Hanks' → 'tomhanks'  (prevents stop-word splitting)
    """
    try:
        items = ast.literal_eval(str(text))
        names = [i["name"] for i in items if isinstance(i, dict) and "name" in i]
        if limit:
            names = names[:limit]
        return " ".join(n.replace(" ", "").lower() for n in names)
    except Exception:
        return ""

def parse_director(text):
    try:
        crew = ast.literal_eval(str(text))
        for m in crew:
            if isinstance(m, dict) and m.get("job") == "Director":
                return m.get("name", "").replace(" ", "").lower()
        return ""
    except Exception:
        return ""

In [11]:
# =========================================================
# MERGE credits.csv
# =========================================================

has_credits = os.path.exists(CREDITS)
if has_credits:
    print("Merging credits.csv ...")
    cred = pd.read_csv(CREDITS)
    cred["id"] = pd.to_numeric(cred["id"], errors="coerce")
    cred = cred.dropna(subset=["id"])
    cred["id"] = cred["id"].astype(int)
    cred["cast_clean"]     = cred["cast"].fillna("[]").apply(lambda x: parse_names(x, limit=5))
    cred["director_clean"] = cred["crew"].fillna("[]").apply(parse_director)
    df = df.merge(cred[["id","cast_clean","director_clean"]], on="id", how="left")
    df["cast_clean"]     = df["cast_clean"].fillna("")
    df["director_clean"] = df["director_clean"].fillna("")
    print(f"  Sample cast: {df['cast_clean'].iloc[0]}")
else:
    print("  WARNING: credits.csv not found — cast/director signals missing.")
    df["cast_clean"]     = ""
    df["director_clean"] = ""

Merging credits.csv ...
  Sample cast: tomhanks timallen donrickles jimvarney wallaceshawn


In [12]:
# =========================================================
# MERGE keywords.csv
# =========================================================

has_keywords = os.path.exists(KEYWORDS)
if has_keywords:
    print("Merging keywords.csv ...")
    kw = pd.read_csv(KEYWORDS)
    kw["id"] = pd.to_numeric(kw["id"], errors="coerce")
    kw = kw.dropna(subset=["id"])
    kw["id"] = kw["id"].astype(int)
    kw["keywords_clean"] = kw["keywords"].fillna("[]").apply(parse_names)
    df = df.merge(kw[["id","keywords_clean"]], on="id", how="left")
    df["keywords_clean"] = df["keywords_clean"].fillna("")
    print(f"  Sample keywords: {df['keywords_clean'].iloc[0]}")
else:
    print("  WARNING: keywords.csv not found — keyword signals missing.")
    df["keywords_clean"] = ""

Merging keywords.csv ...
  Sample keywords: jealousy toy boy friendship friends rivalry boynextdoor newtoy toycomestolife


In [13]:
# =========================================================
# GENRES (stored clean in df so app never re-parses raw dicts)
# =========================================================

df["genres_clean"] = df["genres"].apply(parse_names)

In [14]:
# =========================================================
# SORT + DEDUP TITLES
# =========================================================

df = df.sort_values("popularity", ascending=False)
df = df.drop_duplicates(subset="title", keep="first")
df = df.reset_index(drop=True)
print(f"\n  Final: {len(df):,} unique titles")


  Final: 42,277 unique titles


In [15]:
# =========================================================
# BUILD TAGS — use list join, NOT string multiplication
# =========================================================

def build_tags(row):
    parts = []

    genre    = row.get("genres_clean", "").strip()
    keywords = row.get("keywords_clean", "").strip()
    cast     = row.get("cast_clean", "").strip()
    director = row.get("director_clean", "").strip()
    tagline  = row.get("tagline", "").strip()
    overview = row.get("overview", "").strip()

    # Repeat by extending the list with copies — NOT string multiplication
    if genre:    parts.extend([genre]    * 3)   # "action action action"
    if keywords: parts.extend([keywords] * 3)   # "disaster romance disaster romance ..."
    if director: parts.extend([director] * 3)   # "jamescameron jamescameron jamescameron"
    if cast:     parts.append(cast)             # cast once (already rare tokens)
    if tagline:  parts.append(tagline)
    if overview: parts.append(overview)         # overview once

    return " ".join(parts)

df["tags"] = df.apply(build_tags, axis=1)


In [16]:
# =========================================================
# NLP PREPROCESSING
# =========================================================

nltk.download("stopwords", quiet=True)
nltk.download("wordnet",   quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


In [17]:
# Extended stop-words — covers high-freq low-signal overview words
MOVIE_STOP = {
    "year","years","day","days","night","time","later","soon",
    "new","old","long","story","film","movie","tell","follow",
    "base","true","life","world","man","woman","one","two",
    "three","set","series","part","take","make","come","get",
    "see","know","find","use","try","want","give","live","help",
    "leave","turn","keep","put","bring","start","show","play",
    "run","hold","move","return","lead","young","local","small",
    "big","great","little","good","bad","best","real","first",
    "last","next","never","ever","back","away",
}

ALL_STOP = set(stopwords.words("english")) | MOVIE_STOP
LEMMA    = WordNetLemmatizer()

def preprocess(text):
    text  = str(text).lower()
    text  = re.sub(r"[^a-z\s]", " ", text)
    words = [LEMMA.lemmatize(w) for w in text.split()
             if len(w) > 2 and w not in ALL_STOP]
    return " ".join(words)

print("Preprocessing tags ...")
df["tags"] = df["tags"].apply(preprocess)

Preprocessing tags ...


In [18]:
# =========================================================
# INDICES
# =========================================================

indices = pd.Series(df.index, index=df["title"].str.lower()).drop_duplicates()


In [19]:
# =========================================================
# TF-IDF
# =========================================================

print("Fitting TF-IDF ...")
tfidf = TfidfVectorizer(
    max_features=25000,
    ngram_range=(1, 2),
    stop_words="english",
    sublinear_tf=True,   # log(1+tf) — makes the ×3 boost work correctly
    min_df=3,
    max_df=0.85,
)
tfidf_matrix = tfidf.fit_transform(df["tags"])
print(f"  Matrix: {tfidf_matrix.shape}")

Fitting TF-IDF ...
  Matrix: (42277, 25000)


In [20]:
# =========================================================
# EMBEDDINGS (sentence-transformers)
# =========================================================

if os.path.exists(EMB_PATH):
    print("Loading cached embeddings ...")
    with open(EMB_PATH, "rb") as f:
        embeddings = pickle.load(f)
else:
    try:
        from sentence_transformers import SentenceTransformer
        print("Generating embeddings")
        # model = SentenceTransformer("all-MiniLM-L6-v2")
        model = SentenceTransformer('paraphrase-MiniLM-L3-v2')
        embeddings = model.encode(
            df["tags"].tolist(),
            batch_size=128,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        with open(EMB_PATH, "wb") as f:
            pickle.dump(embeddings, f)
        print("  Embeddings saved.")
    except ImportError:
        print("  sentence-transformers not installed.")
        print("  Run: pip install sentence-transformers")
        print("  Saving zeros placeholder — TF-IDF only mode.")
        embeddings = np.zeros((len(df), 1))
        with open(EMB_PATH, "wb") as f:
            pickle.dump(embeddings, f)

Loading cached embeddings ...


In [21]:
# =========================================================
# SANITY CHECK
# =========================================================

def get_top_terms(title, n=10):
    t = title.lower()
    if t not in indices:
        print(f"  '{title}' not found"); return
    idx      = indices[t]
    features = tfidf.get_feature_names_out()
    vec      = tfidf_matrix[idx].toarray().flatten()
    top      = vec.argsort()[::-1][:n]
    print(f"\nTop {n} TF-IDF terms for '{df.iloc[idx]['title']}':")
    for i in top:
        if vec[i] > 0:
            print(f"  {features[i]:35s}  {vec[i]:.4f}")

def test_recommend(title, n=6):
    t = title.lower()
    if t not in indices:
        print(f"  '{title}' not found"); return
    idx  = indices[t]
    sims = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    top  = sims.argsort()[::-1][1:n+1]
    print(f"\nRecs for '{df.iloc[idx]['title']}':")
    for i in top:
        r = df.iloc[i]
        print(f"  {r['title']:45s}  sim={sims[i]:.3f}  {r['genres_clean']}")

print("\n--- SANITY CHECK ---")
for t in ["titanic", "avatar", "the dark knight", "inception"]:
    if t in indices.index:
        get_top_terms(t, 8)
        test_recommend(t, 5)
        print()


--- SANITY CHECK ---

Top 8 TF-IDF terms for 'Titanic':
  titanic                              0.2696
  oceanliner                           0.2431
  starcrossedlovers                    0.2357
  iceberg                              0.2253
  jamescameron                         0.2213
  salvage                              0.2190
  tragiclove                           0.2190
  classdifferences                     0.2179

Recs for 'Titanic':
  Ghosts of the Abyss                            sim=0.305  documentary
  Titanica                                       sim=0.289  documentary
  Titanic: The Final Word with James Cameron     sim=0.230  documentary history
  A Night to Remember                            sim=0.216  drama action history
  T2 3-D: Battle Across Time                     sim=0.199  


Top 8 TF-IDF terms for 'Avatar':
  powerrelations                       0.2323
  spacecolony                          0.2287
  mindandsoul                          0.2255
  spacewar     

In [23]:
# =========================================================
# SAVE
# =========================================================

os.makedirs(BASE, exist_ok=True)
for name, obj in [
    ("df.pkl",           df),
    ("indices.pkl",      indices),
    ("tfidf_matrix.pkl", tfidf_matrix),
    ("tfidf.pkl",        tfidf),
]:
    with open(os.path.join(BASE, name), "wb") as f:
        pickle.dump(obj, f)
    print(f"  Saved {name}")



  Saved df.pkl
  Saved indices.pkl
  Saved tfidf_matrix.pkl
  Saved tfidf.pkl
